# Revisión R1 — Baselines, sensibilidad ajustada y CKA en VGG-16

Este cuaderno responde al comentario **R1** del editor de *Journal of Imaging*.

El análisis anterior conservó solamente tres resultados escalares para cada comparación original–transformada: distancia euclidiana, similitud coseno y distancia euclidiana relativa. R1 exige conservar ahora los vectores completos para establecer qué similitud produce VGG-16 entre **imágenes originales diferentes**.

El cuaderno realiza cinco tareas:

1. reproduce exactamente la extracción anterior de VGG-16;
2. guarda los embeddings completos de Block1–Block5;
3. verifica que las métricas recalculadas coincidan con el CSV ya publicado en el manuscrito;
4. calcula referencias entre pares aleatorios, de la misma clase y de clases diferentes;
5. calcula sensibilidad ajustada por bloque y CKA entre originales y transformadas.

**Importante:** los resultados anteriores no se sobrescriben. Todo se guarda en una carpeta nueva.

## Antes de ejecutar

En Colab selecciona **Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU** y después ejecuta todas las celdas en orden.

El cuaderno supone que estas carpetas ya existen en Google Drive:

- `2026-0/DatasetOriginalCompleto`
- `2026-0/TrashNet_transformaciones_completo_png`
- `2026-0/Resultados_VGG16_TrashNet_completo_v2_resize224`

Si Colab se desconecta, vuelve a ejecutar el cuaderno. Los archivos de embeddings completos se validan y se omiten automáticamente.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

## 1. Configuración reproducible

In [ ]:
import json
import platform
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torchvision
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import vgg16, VGG16_Weights
from tqdm.auto import tqdm

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

BASE_DIR = Path('/content/gdrive/MyDrive/2026-0')
ORIGINAL_ROOT = BASE_DIR / 'DatasetOriginalCompleto'
TRANSFORMED_ROOT = BASE_DIR / 'TrashNet_transformaciones_completo_png'
OLD_RESULTS_ROOT = BASE_DIR / 'Resultados_VGG16_TrashNet_completo_v2_resize224'
OLD_RESULTS_CSV = OLD_RESULTS_ROOT / 'embeddings_metricas_VGG16_TrashNet_completo_v2.csv'

R1_ROOT = BASE_DIR / 'Revision_R1_Baseline_CKA_VGG16'
EMBEDDING_ROOT = R1_ROOT / 'embeddings'
TRANSFORMED_EMBEDDING_ROOT = EMBEDDING_ROOT / 'transformadas'
TABLE_ROOT = R1_ROOT / 'tablas'
FIGURE_ROOT = R1_ROOT / 'figuras'
for folder in [R1_ROOT, EMBEDDING_ROOT, TRANSFORMED_EMBEDDING_ROOT, TABLE_ROOT, FIGURE_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

ORIGINAL_EMBEDDINGS = EMBEDDING_ROOT / 'embeddings_originales_VGG16_R1.npz'
IMAGE_INDEX_CSV = EMBEDDING_ROOT / 'indice_imagenes_originales_R1.csv'

CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
EXPECTED_COUNTS = {
    'cardboard': 403, 'glass': 501, 'metal': 410,
    'paper': 594, 'plastic': 482, 'trash': 137
}
CONDITIONS = {
    'brightness': {'low': 1.2, 'medium': 1.4, 'high': 1.6},
    'contrast': {'low': 1.2, 'medium': 1.4, 'high': 1.6},
    'rotation': {'low': 10, 'medium': 30, 'high': 60},
    'zoom': {'low': 1.10, 'medium': 1.25, 'high': 1.50},
}
LAYER_INDEX = {
    'Block1': 3,
    'Block2': 8,
    'Block3': 15,
    'Block4': 22,
    'Block5': 29,
}
EXPECTED_DIMENSIONS = {
    'Block1': 64,
    'Block2': 128,
    'Block3': 256,
    'Block4': 512,
    'Block5': 512,
}
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}
BATCH_SIZE = 32
NUM_WORKERS = 2
N_BASELINE_PAIRS = 100_000
N_PLOT_PAIRS = 10_000
EPS = 1e-8
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Dispositivo:', DEVICE)
print('Resultados R1:', R1_ROOT)

### Qué significa la semilla

La semilla `2026` hace que la selección aleatoria de pares sea reproducible. Si otra persona ejecuta el cuaderno con los mismos datos y versiones, seleccionará los mismos pares.

## 2. Inventario ordenado de las 2527 imágenes originales

In [ ]:
def list_images(folder):
    return sorted(
        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS],
        key=lambda p: p.name.lower()
    )

if not ORIGINAL_ROOT.exists():
    raise FileNotFoundError(f'No existe la carpeta de originales: {ORIGINAL_ROOT}')
if not TRANSFORMED_ROOT.exists():
    raise FileNotFoundError(f'No existe la carpeta de transformaciones: {TRANSFORMED_ROOT}')
if not OLD_RESULTS_CSV.exists():
    raise FileNotFoundError(f'No existe el CSV anterior para la validación: {OLD_RESULTS_CSV}')

records = []
ordered_original_paths = []
for class_name in CLASSES:
    class_paths = list_images(ORIGINAL_ROOT / class_name)
    observed = len(class_paths)
    expected = EXPECTED_COUNTS[class_name]
    print(f'{class_name:10s}: {observed:4d}/{expected:4d}')
    if observed != expected:
        raise ValueError(f'Conteo incorrecto en {class_name}: {observed}; se esperaban {expected}.')

    for original_path in class_paths:
        row_index = len(ordered_original_paths)
        ordered_original_paths.append(original_path)
        records.append({
            'row_index': row_index,
            'image_id': original_path.stem,
            'original_filename': original_path.name,
            'class': class_name,
            'original_path': str(original_path),
        })

image_index = pd.DataFrame(records)
if len(image_index) != 2527:
    raise RuntimeError(f'Se encontraron {len(image_index)} imágenes; se esperaban 2527.')
if image_index[['class', 'original_filename']].duplicated().any():
    raise RuntimeError('Existen identificadores duplicados en el índice de originales.')

image_index.to_csv(IMAGE_INDEX_CSV, index=False)
display(image_index.head())
print('Índice guardado:', IMAGE_INDEX_CSV)

## 3. VGG-16 congelada y captura de Block1–Block5

In [ ]:
weights = VGG16_Weights.IMAGENET1K_V1

# Se reproduce exactamente el preprocesamiento del experimento anterior.
# Resize((224,224)) no preserva la relación de aspecto.
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

model = vgg16(weights=weights).to(DEVICE).eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)

activations = {}
handles = []
for block_name, layer_index in LAYER_INDEX.items():
    def save_activation(module, inputs, output, name=block_name):
        activations[name] = output
    handles.append(model.features[layer_index].register_forward_hook(save_activation))

print('Capas capturadas:')
for name, idx in LAYER_INDEX.items():
    print(name, idx, model.features[idx])

Cada tensor de activación tiene dimensiones `canales × alto × ancho`. Igual que en el estudio original, se aplica promedio global sobre alto y ancho. El resultado es un vector por imagen y bloque.

## 4. Funciones de extracción y almacenamiento seguro

In [ ]:
class ImagePathDataset(Dataset):
    def __init__(self, paths):
        self.paths = list(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        with Image.open(path) as image:
            tensor = preprocess(image.convert('RGB'))
        return tensor


def load_npz_arrays(path):
    with np.load(path, allow_pickle=False) as stored:
        return {key: stored[key] for key in stored.files}


def validate_embedding_file(path, expected_rows=2527):
    if not path.exists():
        return False
    try:
        arrays = load_npz_arrays(path)
        if set(arrays) != set(LAYER_INDEX):
            return False
        for block, expected_dim in EXPECTED_DIMENSIONS.items():
            if arrays[block].shape != (expected_rows, expected_dim):
                return False
            if not np.isfinite(arrays[block]).all():
                return False
        return True
    except Exception:
        return False


def extract_embeddings(paths, output_path, description):
    paths = list(paths)
    if validate_embedding_file(output_path, len(paths)):
        print(f'OMITIDO: {output_path.name} ya existe y pasó la validación.')
        return output_path

    dataset = ImagePathDataset(paths)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(NUM_WORKERS > 0),
    )
    collected = {block: [] for block in LAYER_INDEX}

    with torch.inference_mode():
        for batch in tqdm(loader, desc=description):
            batch = batch.to(DEVICE, non_blocking=True)
            activations.clear()
            _ = model(batch)
            for block in LAYER_INDEX:
                pooled = activations[block].mean(dim=(2, 3))
                collected[block].append(pooled.cpu().numpy().astype(np.float32))

    arrays = {block: np.concatenate(parts, axis=0) for block, parts in collected.items()}
    for block, expected_dim in EXPECTED_DIMENSIONS.items():
        expected_shape = (len(paths), expected_dim)
        if arrays[block].shape != expected_shape:
            raise RuntimeError(f'{block}: forma {arrays[block].shape}; se esperaba {expected_shape}.')

    temporary_path = output_path.with_name(output_path.stem + '.tmp.npz')
    np.savez_compressed(temporary_path, **arrays)
    temporary_path.replace(output_path)

    if not validate_embedding_file(output_path, len(paths)):
        raise RuntimeError(f'El archivo guardado no pasó la validación: {output_path}')
    print('GUARDADO:', output_path)
    return output_path

## 5. Extraer y guardar embeddings de las imágenes originales

In [ ]:
extract_embeddings(
    ordered_original_paths,
    ORIGINAL_EMBEDDINGS,
    'Embeddings originales'
)
original_embeddings = load_npz_arrays(ORIGINAL_EMBEDDINGS)

for block in LAYER_INDEX:
    print(block, original_embeddings[block].shape, original_embeddings[block].dtype)

Este archivo contiene las “coordenadas” internas de cada imagen en los cinco espacios representacionales. El CSV `indice_imagenes_originales_R1.csv` permite identificar a qué imagen corresponde cada fila.

## 6. Extraer embeddings transformados por condición

In [ ]:
transformed_embedding_paths = {}

for transformation, levels in CONDITIONS.items():
    for level, value in levels.items():
        condition = f'{transformation}_{level}'
        transformed_paths = []
        for _, row in image_index.iterrows():
            transformed_path = (
                TRANSFORMED_ROOT / condition / row['class'] /
                f"{Path(row['original_filename']).stem}.png"
            )
            if not transformed_path.exists():
                raise FileNotFoundError(f'Falta: {transformed_path}')
            transformed_paths.append(transformed_path)

        output_path = TRANSFORMED_EMBEDDING_ROOT / f'embeddings_{condition}.npz'
        extract_embeddings(transformed_paths, output_path, f'Embeddings {condition}')
        transformed_embedding_paths[(transformation, level)] = output_path

print('Condiciones completas:', len(transformed_embedding_paths))

## 7. Auditoría contra las métricas del experimento anterior

In [ ]:
def vector_metrics_numpy(original, transformed, eps=1e-8):
    difference = original - transformed
    euclidean = np.linalg.norm(difference, axis=1)
    original_norm = np.linalg.norm(original, axis=1)
    transformed_norm = np.linalg.norm(transformed, axis=1)
    cosine = np.sum(original * transformed, axis=1) / (
        np.maximum(original_norm, eps) * np.maximum(transformed_norm, eps)
    )
    relative = euclidean / (original_norm + eps)
    return euclidean, cosine, relative


previous_results = pd.read_csv(OLD_RESULTS_CSV)
expected_rows = 2527 * 12 * 5
if len(previous_results) != expected_rows:
    raise RuntimeError(f'El CSV anterior tiene {len(previous_results)} filas; se esperaban {expected_rows}.')

audit_rows = []
for transformation, levels in CONDITIONS.items():
    for level in levels:
        transformed = load_npz_arrays(transformed_embedding_paths[(transformation, level)])
        previous_condition = previous_results[
            (previous_results['transformation'] == transformation) &
            (previous_results['level'] == level)
        ]

        for block in LAYER_INDEX:
            euclidean, cosine, relative = vector_metrics_numpy(
                original_embeddings[block], transformed[block], EPS
            )
            recalculated = image_index[['image_id', 'class']].copy()
            recalculated['euclidean_new'] = euclidean
            recalculated['cosine_new'] = cosine
            recalculated['relative_new'] = relative

            previous_block = previous_condition[previous_condition['block'] == block][[
                'image_id', 'class', 'euclidean_distance',
                'cosine_similarity', 'relative_euclidean_distance'
            ]]
            merged = recalculated.merge(
                previous_block,
                on=['image_id', 'class'],
                how='inner',
                validate='one_to_one'
            )
            if len(merged) != 2527:
                raise RuntimeError(f'Validación incompleta en {transformation}-{level}-{block}.')

            audit_rows.append({
                'transformation': transformation,
                'level': level,
                'block': block,
                'n': len(merged),
                'max_abs_error_euclidean': float(np.max(np.abs(
                    merged['euclidean_new'] - merged['euclidean_distance']))),
                'max_abs_error_cosine': float(np.max(np.abs(
                    merged['cosine_new'] - merged['cosine_similarity']))),
                'max_abs_error_relative': float(np.max(np.abs(
                    merged['relative_new'] - merged['relative_euclidean_distance']))),
            })

reproduction_audit = pd.DataFrame(audit_rows)
# Las operaciones en GPU pueden diferir unas pocas unidades en los últimos
# decimales debido al orden de acumulación de punto flotante. La distancia
# euclidiana acumula hasta 512 componentes, por lo que requiere una tolerancia
# absoluta ligeramente mayor que el coseno y la distancia relativa.
AUDIT_ATOL_EUCLIDEAN = 2e-5
AUDIT_ATOL_COSINE = 1e-6
AUDIT_ATOL_RELATIVE = 3e-6

reproduction_audit['passed_euclidean'] = (
    reproduction_audit['max_abs_error_euclidean'] <= AUDIT_ATOL_EUCLIDEAN
)
reproduction_audit['passed_cosine'] = (
    reproduction_audit['max_abs_error_cosine'] <= AUDIT_ATOL_COSINE
)
reproduction_audit['passed_relative'] = (
    reproduction_audit['max_abs_error_relative'] <= AUDIT_ATOL_RELATIVE
)
reproduction_audit['passed_all'] = reproduction_audit[[
    'passed_euclidean', 'passed_cosine', 'passed_relative'
]].all(axis=1)
audit_path = TABLE_ROOT / '01_auditoria_reproduccion_metricas.csv'
reproduction_audit.to_csv(audit_path, index=False, float_format='%.10g')
display(reproduction_audit)
print('Tolerancias absolutas:')
print('  Euclidiana:', AUDIT_ATOL_EUCLIDEAN)
print('  Coseno:', AUDIT_ATOL_COSINE)
print('  Relativa:', AUDIT_ATOL_RELATIVE)
print('Todas las condiciones reproducidas:', reproduction_audit['passed_all'].all())
if not reproduction_audit['passed_all'].all():
    failed = reproduction_audit.loc[~reproduction_audit['passed_all']]
    display(failed)
    raise RuntimeError('La nueva extracción excede las tolerancias numéricas definidas.')

Esta auditoría es indispensable. No continuamos con R1 si la nueva extracción no reproduce las métricas utilizadas en el manuscrito.

## 8. Pares de referencia solicitados por R1

Se crean tres conjuntos reproducibles de pares de imágenes originales diferentes:

- **Random:** pares seleccionados sin considerar la clase.
- **Same class:** dos imágenes distintas de la misma clase.
- **Different class:** imágenes de clases diferentes.

Se utiliza el mismo conjunto de índices en los cinco bloques para que las diferencias observadas se deban al bloque y no a un cambio de imágenes emparejadas.

In [ ]:
n_images = len(image_index)
pair_i, pair_j = np.triu_indices(n_images, k=1)
labels = image_index['class'].to_numpy()
same_class_mask = labels[pair_i] == labels[pair_j]

eligible_positions = {
    'random': np.arange(len(pair_i)),
    'same_class': np.flatnonzero(same_class_mask),
    'different_class': np.flatnonzero(~same_class_mask),
}

rng = np.random.default_rng(SEED)
selected_positions = {}
for pair_type, eligible in eligible_positions.items():
    sample_size = min(N_BASELINE_PAIRS, len(eligible))
    selected_positions[pair_type] = rng.choice(eligible, size=sample_size, replace=False)
    print(f'{pair_type:16s}: elegibles={len(eligible):,}; seleccionados={sample_size:,}')

In [ ]:
def describe_values(values):
    values = np.asarray(values, dtype=np.float64)
    return {
        'mean': float(np.mean(values)),
        'sd': float(np.std(values, ddof=1)),
        'median': float(np.median(values)),
        'q2_5': float(np.quantile(values, 0.025)),
        'q25': float(np.quantile(values, 0.25)),
        'q75': float(np.quantile(values, 0.75)),
        'q97_5': float(np.quantile(values, 0.975)),
        'minimum': float(np.min(values)),
        'maximum': float(np.max(values)),
    }


baseline_rows = []
baseline_samples = {}
for block_index, block in enumerate(LAYER_INDEX, start=1):
    embeddings = original_embeddings[block].astype(np.float64)
    normalized = embeddings / np.maximum(np.linalg.norm(embeddings, axis=1, keepdims=True), EPS)

    for pair_type, selected in selected_positions.items():
        i = pair_i[selected]
        j = pair_j[selected]
        similarities = np.sum(normalized[i] * normalized[j], axis=1).astype(np.float32)
        baseline_samples[f'{block}_{pair_type}'] = similarities

        row = {
            'block': block,
            'block_index': block_index,
            'pair_type': pair_type,
            'eligible_pairs': len(eligible_positions[pair_type]),
            'sampled_pairs': len(similarities),
        }
        row.update(describe_values(similarities))
        baseline_rows.append(row)

baseline_summary = pd.DataFrame(baseline_rows)
baseline_summary_path = TABLE_ROOT / '02_baseline_cosine_pair_summary.csv'
baseline_summary.to_csv(baseline_summary_path, index=False, float_format='%.10f')
np.savez_compressed(EMBEDDING_ROOT / 'muestras_pares_baseline_R1.npz', **baseline_samples)
display(baseline_summary)

## 9. Figura de las similitudes de referencia

In [ ]:
plot_rng = np.random.default_rng(SEED + 1)
plot_frames = []
pair_labels = {
    'random': 'Random pairs',
    'same_class': 'Same-class pairs',
    'different_class': 'Different-class pairs',
}
for block in LAYER_INDEX:
    for pair_type in pair_labels:
        values = baseline_samples[f'{block}_{pair_type}']
        if len(values) > N_PLOT_PAIRS:
            values = plot_rng.choice(values, size=N_PLOT_PAIRS, replace=False)
        plot_frames.append(pd.DataFrame({
            'Block': block,
            'Pair type': pair_labels[pair_type],
            'Cosine similarity': values,
        }))

baseline_plot_data = pd.concat(plot_frames, ignore_index=True)
sns.set_theme(style='whitegrid', context='paper')
plt.figure(figsize=(10.5, 5.8))
ax = sns.boxplot(
    data=baseline_plot_data,
    x='Block', y='Cosine similarity', hue='Pair type',
    showfliers=False
)
ax.set_title('Baseline cosine similarity between different original images')
ax.legend(title='')
plt.tight_layout()
baseline_figure_path = FIGURE_ROOT / 'Figure_R1_baseline_cosine_by_block.png'
plt.savefig(baseline_figure_path, dpi=300, bbox_inches='tight')
plt.show()
print('Figura:', baseline_figure_path)

## 10. Sensibilidad ajustada por la referencia de cada bloque

Para conservar una interpretación en la que valores mayores significan mayor alteración, se define:

\[
A_{\mathrm{adjusted}}=\frac{1-S_{\mathrm{transformed}}}{1-B_{\mathrm{block}}},
\]

donde \(S_{\mathrm{transformed}}\) es el coseno original–transformada y
\(B_{\mathrm{block}}\) es la mediana de los pares aleatorios del bloque.

- 0: sin cambio respecto del original;
- 1: cambio comparable con la separación típica entre dos imágenes originales diferentes;
- mayor que 1: cambio superior a esa referencia.

No se recortan los valores superiores a 1 porque contienen información.

In [ ]:
random_baseline = (
    baseline_summary[baseline_summary['pair_type'] == 'random']
    .set_index('block')['median']
    .to_dict()
)

adjusted = previous_results.copy()
adjusted['random_pair_baseline_median'] = adjusted['block'].map(random_baseline)
adjusted['baseline_adjusted_sensitivity'] = (
    (1.0 - adjusted['cosine_similarity']) /
    (1.0 - adjusted['random_pair_baseline_median'])
)
adjusted['baseline_adjusted_retention'] = 1.0 - adjusted['baseline_adjusted_sensitivity']

if not np.isfinite(adjusted['baseline_adjusted_sensitivity']).all():
    raise RuntimeError('Se detectaron valores no finitos en la sensibilidad ajustada.')

adjusted_path = TABLE_ROOT / '03_metricas_individuales_con_sensibilidad_ajustada.csv'
adjusted.to_csv(adjusted_path, index=False, float_format='%.10f')

adjusted_summary = (
    adjusted.groupby(['transformation', 'level', 'value', 'block'], as_index=False)
    .agg(
        n=('baseline_adjusted_sensitivity', 'size'),
        adjusted_mean=('baseline_adjusted_sensitivity', 'mean'),
        adjusted_sd=('baseline_adjusted_sensitivity', 'std'),
        adjusted_median=('baseline_adjusted_sensitivity', 'median'),
        adjusted_q2_5=('baseline_adjusted_sensitivity', lambda x: x.quantile(0.025)),
        adjusted_q97_5=('baseline_adjusted_sensitivity', lambda x: x.quantile(0.975)),
    )
)
adjusted_summary['level_order'] = adjusted_summary['level'].map({'low': 0, 'medium': 1, 'high': 2})
adjusted_summary['block_index'] = adjusted_summary['block'].str.extract(r'(\d+)').astype(int)
adjusted_summary = adjusted_summary.sort_values(
    ['transformation', 'level_order', 'block_index']
).drop(columns='level_order')

adjusted_summary_path = TABLE_ROOT / '04_resumen_sensibilidad_ajustada.csv'
adjusted_summary.to_csv(adjusted_summary_path, index=False, float_format='%.10f')
display(adjusted_summary)

In [ ]:
transformations = ['brightness', 'contrast', 'rotation', 'zoom']
levels = ['low', 'medium', 'high']
colors = {'low': '#2ca02c', 'medium': '#ff7f0e', 'high': '#d62728'}

fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
axes = axes.ravel()
for panel, (ax, transformation) in enumerate(zip(axes, transformations)):
    subset = adjusted_summary[adjusted_summary['transformation'] == transformation]
    for level in levels:
        line = subset[subset['level'] == level].sort_values('block_index')
        ax.plot(
            line['block_index'], line['adjusted_mean'], marker='o',
            label=level.capitalize(), color=colors[level]
        )
    ax.axhline(1.0, color='black', linestyle='--', linewidth=0.9, alpha=0.7)
    ax.set_title(f"({chr(97 + panel)}) {transformation.capitalize()}")
    ax.set_xticks(range(1, 6), [f'Block{i}' for i in range(1, 6)])
    ax.set_ylabel('Baseline-adjusted sensitivity')
    ax.grid(alpha=0.25)

handles_plot, labels_plot = axes[0].get_legend_handles_labels()
fig.legend(handles_plot, labels_plot, loc='upper center', ncol=3, frameon=False)
fig.suptitle('Layer-wise sensitivity adjusted by the random-pair baseline', y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.95])
adjusted_figure_path = FIGURE_ROOT / 'Figure_R1_adjusted_sensitivity_by_block.png'
fig.savefig(adjusted_figure_path, dpi=300, bbox_inches='tight')
plt.show()
print('Figura:', adjusted_figure_path)

## 11. CKA lineal centrado

CKA compara la estructura del conjunto completo de imágenes. Evalúa si las relaciones representacionales entre las 2527 imágenes originales se conservan después de una transformación.

- valor próximo a 1: mayor conservación estructural;
- valor menor: mayor reorganización representacional.

CKA complementa, pero no sustituye, la comparación individual mediante coseno.

In [ ]:
def linear_cka_centered(x, y, eps=1e-12):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    x = x - x.mean(axis=0, keepdims=True)
    y = y - y.mean(axis=0, keepdims=True)

    cross = x.T @ y
    xx = x.T @ x
    yy = y.T @ y
    numerator = np.sum(cross * cross)
    denominator = np.sqrt(np.sum(xx * xx) * np.sum(yy * yy))
    if denominator <= eps:
        return np.nan
    return float(numerator / denominator)


cka_rows = []
for transformation, levels_dict in CONDITIONS.items():
    for level, value in levels_dict.items():
        transformed = load_npz_arrays(transformed_embedding_paths[(transformation, level)])
        for block_index, block in enumerate(LAYER_INDEX, start=1):
            cka_rows.append({
                'transformation': transformation,
                'level': level,
                'value': value,
                'block': block,
                'block_index': block_index,
                'n_images': 2527,
                'linear_CKA': linear_cka_centered(
                    original_embeddings[block], transformed[block]
                ),
            })

cka_results = pd.DataFrame(cka_rows)
cka_results['level_order'] = cka_results['level'].map({'low': 0, 'medium': 1, 'high': 2})
cka_results = cka_results.sort_values(
    ['transformation', 'level_order', 'block_index']
).drop(columns='level_order')
cka_path = TABLE_ROOT / '05_CKA_lineal_centrado_por_condicion.csv'
cka_results.to_csv(cka_path, index=False, float_format='%.10f')
display(cka_results)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
axes = axes.ravel()
for panel, (ax, transformation) in enumerate(zip(axes, transformations)):
    subset = cka_results[cka_results['transformation'] == transformation]
    for level in levels:
        line = subset[subset['level'] == level].sort_values('block_index')
        ax.plot(
            line['block_index'], line['linear_CKA'], marker='o',
            label=level.capitalize(), color=colors[level]
        )
    ax.set_title(f"({chr(97 + panel)}) {transformation.capitalize()}")
    ax.set_xticks(range(1, 6), [f'Block{i}' for i in range(1, 6)])
    ax.set_ylabel('Centered linear CKA')
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.25)

handles_plot, labels_plot = axes[0].get_legend_handles_labels()
fig.legend(handles_plot, labels_plot, loc='upper center', ncol=3, frameon=False)
fig.suptitle('Representational alignment between original and transformed images', y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.95])
cka_figure_path = FIGURE_ROOT / 'Figure_R1_CKA_by_block.png'
fig.savefig(cka_figure_path, dpi=300, bbox_inches='tight')
plt.show()
print('Figura:', cka_figure_path)

## 12. Resumen integrado para interpretar R1

In [ ]:
integrated = adjusted_summary.merge(
    cka_results[['transformation', 'level', 'block', 'linear_CKA']],
    on=['transformation', 'level', 'block'],
    how='left',
    validate='one_to_one'
)
integrated['random_pair_baseline_median'] = integrated['block'].map(random_baseline)
integrated_path = TABLE_ROOT / '06_resumen_integrado_R1.csv'
integrated.to_csv(integrated_path, index=False, float_format='%.10f')
display(integrated)

## 13. Metadatos y auditoría final

In [ ]:
r1_environment = {
    'analysis': 'R1 baseline-adjusted VGG16 representational sensitivity and centered linear CKA',
    'seed': SEED,
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'device': str(DEVICE),
    'weights': 'VGG16_Weights.IMAGENET1K_V1',
    'preprocessing': str(preprocess),
    'layer_indices': LAYER_INDEX,
    'embedding_dimensions': EXPECTED_DIMENSIONS,
    'total_original_images': len(image_index),
    'baseline_pairs_per_type': N_BASELINE_PAIRS,
    'baseline_pair_types': ['random', 'same_class', 'different_class'],
    'baseline_for_adjustment': 'median cosine similarity of seeded unrestricted random pairs within each block',
    'adjusted_sensitivity_formula': '(1 - original_transformed_cosine) / (1 - random_pair_block_median)',
    'cka': 'centered linear CKA',
}
metadata_path = R1_ROOT / 'entorno_y_metodo_R1.json'
metadata_path.write_text(json.dumps(r1_environment, indent=2), encoding='utf-8')

expected_outputs = [
    ORIGINAL_EMBEDDINGS,
    IMAGE_INDEX_CSV,
    TABLE_ROOT / '01_auditoria_reproduccion_metricas.csv',
    TABLE_ROOT / '02_baseline_cosine_pair_summary.csv',
    TABLE_ROOT / '03_metricas_individuales_con_sensibilidad_ajustada.csv',
    TABLE_ROOT / '04_resumen_sensibilidad_ajustada.csv',
    TABLE_ROOT / '05_CKA_lineal_centrado_por_condicion.csv',
    TABLE_ROOT / '06_resumen_integrado_R1.csv',
    FIGURE_ROOT / 'Figure_R1_baseline_cosine_by_block.png',
    FIGURE_ROOT / 'Figure_R1_adjusted_sensitivity_by_block.png',
    FIGURE_ROOT / 'Figure_R1_CKA_by_block.png',
    metadata_path,
]
expected_outputs.extend(transformed_embedding_paths.values())

audit = pd.DataFrame({
    'path': [str(path) for path in expected_outputs],
    'exists': [path.exists() for path in expected_outputs],
    'size_bytes': [path.stat().st_size if path.exists() else 0 for path in expected_outputs],
})
display(audit)
if not audit['exists'].all():
    raise RuntimeError('Faltan resultados de R1. Revisa la tabla de auditoría.')

print('R1 COMPLETO')
print('Carpeta:', R1_ROOT)

## Cómo interpretar los resultados cuando termine

No debe redactarse la respuesta al editor antes de inspeccionar las tres tablas y figuras principales:

1. `02_baseline_cosine_pair_summary.csv`: muestra cuánto se parecen imágenes originales diferentes en cada bloque.
2. `04_resumen_sensibilidad_ajustada.csv`: muestra el cambio original–transformada relativo a la escala propia de cada bloque.
3. `05_CKA_lineal_centrado_por_condicion.csv`: muestra cuánto se conserva la organización del conjunto completo.

Después de ejecutar el cuaderno, comparte la carpeta `Revision_R1_Baseline_CKA_VGG16` o, como mínimo, las seis tablas CSV y las tres figuras. Con esos resultados se determinará si la conclusión sobre profundidad se mantiene, se restringe o debe modificarse.